# Stream Encryption: ChaCha20 and Keystream Safety

These examples use the running scenario of St. Isidore Hospital. They are teaching examples: understand the mechanism, then prefer well-reviewed libraries and current protocols in production.

## Goal

A stream cipher generates a keystream and combines it with plaintext. The key lesson is nonce uniqueness: reusing the same key and nonce can reveal relationships between messages.

In [ ]:
from Crypto.Cipher import ChaCha20
from Crypto.Random import get_random_bytes

key = get_random_bytes(32)  # ChaCha20 uses a 256-bit key in this library.
nonce = get_random_bytes(8)  # The nonce selects a unique keystream for this message.
msg = b"Telemetry: infusion pump dose=4.0 ml/h"

cipher = ChaCha20.new(key=key, nonce=nonce)  # Create the keystream generator.
ct = cipher.encrypt(msg)  # Encryption XORs plaintext with the keystream.
plain = ChaCha20.new(key=key, nonce=nonce).decrypt(ct)  # Decryption XORs with the same keystream.
print(ct.hex())
print(plain)

In [ ]:
def xor_bytes(a, b):
    # XOR two byte strings up to the length of the shorter one.
    return bytes(x ^ y for x, y in zip(a, b))

m1 = b"Lab result patient 2048: potassium normal"
m2 = b"Lab result patient 2048: potassium urgent"
bad_nonce = b"12345678"  # Deliberately reused: this is the mistake.

c1 = ChaCha20.new(key=key, nonce=bad_nonce).encrypt(m1)  # First message under reused keystream.
c2 = ChaCha20.new(key=key, nonce=bad_nonce).encrypt(m2)  # Second message under the same keystream.
print("XOR ciphertexts equals XOR plaintexts:")
print(xor_bytes(c1, c2) == xor_bytes(m1, m2))
print(xor_bytes(c1, c2))